In [4]:
################################# TOPIC 5 - GROUP BY AND AGGREGATION #########################################

In [5]:
# Groupby and aggregation - Summarizing metrics per server, per priority, per time window

In [6]:
# LOAD DATASET TO WORK ON
import pandas as pd
import numpy as np

# ── Load all three datasets ────────────────────────────────────────────
df_metrics = pd.read_csv(
    "server_metrics.csv",
    parse_dates=["timestamp"],
    dtype={"server_id": "category"},
    na_values=["N/A", "null", "—"]
)

df_tickets = pd.read_csv(
    "incidents.csv",
    parse_dates=["created_at", "resolved_at"]
)

df_logs = pd.read_csv(
    "app_logs.csv",
    parse_dates=["timestamp"]
)

# ── Confirm shapes ─────────────────────────────────────────────────────
print(f"df_metrics : {df_metrics.shape}")
print(f"df_tickets : {df_tickets.shape}")
print(f"df_logs    : {df_logs.shape}")

df_metrics : (525, 7)
df_tickets : (213, 7)
df_logs    : (315, 5)


In [7]:
# Basic groupby + single aggregation
df_metrics.groupby("server_id", observed=True)["cpu_pct"].mean()

server_id
srv-01    59.926667
srv-02    59.931429
srv-03    58.565714
srv-04    59.409524
srv-05    62.332381
Name: cpu_pct, dtype: float64

In [8]:
# Multiple aggregations - agg()
# on one column
df_metrics.groupby("server_id", observed=True)["cpu_pct"].agg(["mean", "max", "min", "std"]).round(2)

,mean,max,min,std
server_id,,,,
srv-01,59.93,98.4,21.8,24.37
srv-02,59.93,98.2,20.7,23.06
srv-03,58.57,98.5,20.9,23.10
srv-04,59.41,98.6,20.6,25.00
srv-05,62.33,96.7,20.6,22.01


In [9]:
# Multiple stats on one column
df_metrics.groupby("server_id", observed=True)["cpu_pct"].agg(["mean", "max", "min", "std"]).round(2)

,mean,max,min,std
server_id,,,,
srv-01,59.93,98.4,21.8,24.37
srv-02,59.93,98.2,20.7,23.06
srv-03,58.57,98.5,20.9,23.10
srv-04,59.41,98.6,20.6,25.00
srv-05,62.33,96.7,20.6,22.01


In [10]:
# Naming each aggregation explicitly
df_metrics.groupby("server_id", observed=True).agg(
    avg_cpu = ("cpu_pct", "mean"),
    max_cpu = ("cpu_pct", "max"),
    avg_memory = ("memory_pct", "mean"),
    avg_response = ("response_ms", "mean"),
    total_records = ("cpu_pct", "count")
).round(2)

,avg_cpu,max_cpu,avg_memory,avg_response,total_records
server_id,,,,,
srv-01,59.93,98.4,62.51,643.57,105
srv-02,59.93,98.2,64.46,633.02,105
srv-03,58.57,98.5,65.09,622.77,105
srv-04,59.41,98.6,62.21,632.40,105
srv-05,62.33,96.7,62.10,590.14,105


In [11]:
# Groupby on multiple columns
# Average cpu per server per status
df_metrics.groupby(["server_id", "status"], observed=True)["cpu_pct"].mean().round(2)

server_id  status  
srv-01     critical    58.96
           ok          59.29
           warning     62.57
srv-02     critical    61.67
           ok          57.60
           warning     73.05
srv-03     critical    55.89
           ok          57.33
           warning     68.10
srv-04     critical    62.85
           ok          60.21
           warning     56.25
srv-05     critical    50.13
           ok          61.36
           warning     70.87
Name: cpu_pct, dtype: float64

In [12]:
# Groupby on incidents - ticket analysis
# Count tickets per category
df_tickets.groupby("category")["ticket_id"].count().sort_values(ascending=False)

category
CPU High        50
Disk Full       46
Service Down    46
Memory Leak     36
Network Lag     35
Name: ticket_id, dtype: int64

In [13]:
# Resolution rate per category
df_tickets.groupby("category").agg(
    total = ("ticket_id", "count"),
    resolved = ("resolved", "sum"),
).assign(
    resolution_rate = lambda x:(x["resolved"] / x["total"] * 100 ).round(1)
).sort_values("resolution_rate")

,total,resolved,resolution_rate
category,,,
Network Lag,35,19,54.3
Service Down,46,25,54.3
Disk Full,46,26,56.5
Memory Leak,36,22,61.1
CPU High,50,33,66.0


In [14]:
# Groupby on logs - error rate analysis
# log count per server per log level
df_logs.groupby(["server_id", "log_level"])["message"].count().unstack(fill_value=0)

log_level,CRITICAL,ERROR,INFO,WARNING
server_id,,,,
srv-01,7,5,26,7
srv-02,8,14,32,18
srv-03,6,11,30,19
srv-04,10,6,27,20
srv-05,10,11,34,14


In [15]:
# Error rate per server - ERROR + CRITICAL as % of total logs
log_counts = df_logs.groupby("server_id")["log_level"].value_counts(normalize=True).mul(100).round(1).unstack(fill_value=0)
log_counts["error_rate"] = log_counts.get("ERROR", 0) + log_counts.get("CRITICAL", 0)
log_counts[["ERROR", "CRITICAL", "error_rate"]].sort_values("error_rate", ascending=False)

log_level,ERROR,CRITICAL,error_rate
server_id,,,
srv-02,19.4,11.1,30.5
srv-05,15.9,14.5,30.4
srv-01,11.1,15.6,26.7
srv-03,16.7,9.1,25.8
srv-04,9.5,15.9,25.4


In [17]:
# transform()
# Add fleet average cpu as a column - for comparison per row 
df_metrics["fleet_avg_cpu"] = df_metrics.groupby("server_id", observed=True)["cpu_pct"].transform("mean")
print(df_metrics["fleet_avg_cpu"])

0      59.926667
1      59.931429
2      58.565714
3      59.409524
4      62.332381
         ...    
520    59.926667
521    59.931429
522    58.565714
523    59.409524
524    62.332381
Name: fleet_avg_cpu, Length: 525, dtype: float64


In [18]:
# How far is each reading from its server's average ? 
df_metrics["cpu_deviation"] = df_metrics["cpu_pct"] - df_metrics["fleet_avg_cpu"]
print(df_metrics[["server_id", "cpu_pct", "fleet_avg_cpu", "cpu_deviation"]].head(10))

  server_id  cpu_pct  fleet_avg_cpu  cpu_deviation
0    srv-01     49.6      59.926667     -10.326667
1    srv-02     32.3      59.931429     -27.631429
2    srv-03     21.6      58.565714     -36.965714
3    srv-04     34.5      59.409524     -24.909524
4    srv-05     68.3      62.332381       5.967619
5    srv-01     82.0      59.926667      22.073333
6    srv-02     68.0      59.931429       8.068571
7    srv-03     83.9      58.565714      25.334286
8    srv-04     29.6      59.409524     -29.809524
9    srv-05     72.3      62.332381       9.967619


In [19]:
# filter() - keep only groups meeting a condition

#keep only servers where average CPU > 60
high_load_servers = df_metrics.groupby("server_id", observed=True).filter(lambda x: x["cpu_pct"].mean() > 60)
print(high_load_servers["server_id"].unique())

['srv-05']
Categories (5, object): ['srv-01', 'srv-02', 'srv-03', 'srv-04', 'srv-05']


In [20]:
# resample() preview - hourly aggregation

# Hourly average CPU across entire fleet
df_metrics.set_index("timestamp").resample("1h")["cpu_pct"].mean().head()

timestamp
2026-01-01 00:00:00    41.26
2026-01-01 01:00:00    67.16
2026-01-01 02:00:00    76.80
2026-01-01 03:00:00    59.98
2026-01-01 04:00:00    55.54
Freq: h, Name: cpu_pct, dtype: float64

In [1]:
# Lab — GroupBy + Aggregation
# Task 1 — server_metrics
# For each server compute: mean cpu, max cpu, mean memory, max response_ms, 
# total row count. Use named aggregations. Sort by mean cpu descending. 
# Which server is most loaded on average?

In [ ]:
df_metrics.head(5)

In [ ]:
#df_metrics.columns.tolist()
df_metrics.groupby("server_id", observed=True).agg(
    mean_cpu = ("cpu_pct", "mean"),
    max_cpu = ("cpu_pct", "max"),
    mean_memory = ("memory_pct", "mean"),
    max_response = ("response_ms", "max"),
    total_rows = ("cpu_pct", "count")
).round(2).sort_values("mean_cpu", ascending=False)

In [ ]:
# Task 2 — server_metrics
# Group by server_id AND status. Count rows per combination. 
# Which server has the most critical status readings?

In [ ]:
# Group by server_id AND status. Count rows per combination. 
df_metrics.groupby(["server_id", "status"], observed=True)["cpu_pct"].count()

In [ ]:
# Which server has the most critical status readings?
result = df_metrics.groupby(["server_id", "status"], observed=True)["cpu_pct"].count().reset_index(name="row_count")
print(result)

#Filter critical only and find the max
critical = result[result["status"] == "critical"].sort_values("row_count", ascending=False)
print("\nmax critical\n", critical)
# srv-02 & srv-03 are tied with 7 critical reading each.(max)